# 02 Cleaning


Clean the raw data and write `data/processed/diabetic_data.parquet`.

The logic lives in `eda_diabetes.data.clean`. This notebook shows the effect of each decision,
runs `clean`, checks the result, and writes the parquet file.


In [ ]:
%load_ext autoreload
%autoreload 2
import pandas as pd
import seaborn as sns

from eda_diabetes.data import (
    EXPIRED_HOSPICE_DISCHARGE,
    MISSING_ID_CODES,
    NEAR_CONSTANT_COLUMNS,
    clean,
    load_raw,
)
from eda_diabetes.paths import DATA_PROCESSED

sns.set_theme(style="whitegrid")

In [ ]:
raw = load_raw()
raw.shape

## 1. Drop rows

Three row filters, applied in this order. Count after each one.


In [ ]:
steps = {"raw": len(raw)}
df = raw[~raw["discharge_disposition_id"].isin(EXPIRED_HOSPICE_DISCHARGE)]
steps["drop expired / hospice discharge"] = len(df)
df = df[df["gender"] != "Unknown/Invalid"]
steps["drop gender Unknown/Invalid"] = len(df)
df = df.sort_values("encounter_id").drop_duplicates("patient_nbr", keep="first")
steps["keep first encounter per patient"] = len(df)

row_counts = pd.Series(steps, name="n_rows").to_frame()
row_counts["removed"] = -row_counts["n_rows"].diff().fillna(0).astype(int)
row_counts

- **Expired / hospice:** these patients cannot be readmitted, so they bias the target.
- **Gender `Unknown/Invalid`:** 3 rows, no usable value.
- **One encounter per patient:** the first encounter (lowest `encounter_id`). Repeat encounters of one patient are not independent.
  This loses the later visits of frequent patients.


## 2. Drop columns

- `weight`: 97% missing.
- Near-constant medications: all `No`, or at most 2 other rows. No signal.


In [ ]:
print(raw["weight"].isna().mean().round(3))
raw[NEAR_CONSTANT_COLUMNS].apply(lambda s: s.value_counts().to_dict())

## 3. Missing values

| Column | NaN means | Treatment |
|---|---|---|
| `race`, `payer_code`, `medical_specialty` | unknown | `Unknown` category |
| `max_glu_serum`, `A1Cresult` | test not performed | `None` category |
| `diag_1`, `diag_2`, `diag_3` | no diagnosis | `Unknown` in ICD-9 grouping |
| ID columns with codes such as `NULL`, `Not Mapped` | missing | NaN. Kept as NaN and grouped as `Unknown` |

The ID columns stay NaN in the processed data. Bucket columns (`discharge_group`, `admission_source_group`) show them as `Unknown`.


In [ ]:
hidden = pd.DataFrame(
    {
        col: {
            "codes": sorted(codes),
            "n_rows_in_cohort": int(df[col].isin(codes).sum()),
            "pct": round(df[col].isin(codes).mean() * 100, 1),
        }
        for col, codes in MISSING_ID_CODES.items()
    }
).T
hidden

## 4. Transformations

- `age`: ordered categorical over the 10 bins.
- `readmitted`: ordered categorical `NO < >30 < <30`. New binary target `readmitted_30`.
- `diag_1/2/3`: ICD-9 codes grouped to chapters (Circulatory, Respiratory, Digestive, Diabetes, Genitourinary, Neoplasms, Musculoskeletal, Injury, Other).
- `discharge_group`, `admission_source_group`: ID codes grouped to a few buckets.
- `medical_specialty`: top 10 kept, rest `Other`. `Unknown` stays its own level.
- `n_meds_used`: medications not `No`. `n_meds_changed`: medications `Up` or `Down`.
- `total_visits`: outpatient + emergency + inpatient visits in the previous year.
- Remaining text columns become `category`.


## 5. Run `clean`


In [ ]:
cleaned = clean(raw)
cleaned.shape

In [ ]:
cleaned.dtypes.value_counts()

## 6. Checks


In [ ]:
assert len(cleaned) == steps["keep first encounter per patient"]
assert cleaned["patient_nbr"].is_unique
assert cleaned["encounter_id"].is_unique
assert not cleaned["discharge_disposition_id"].isin(EXPIRED_HOSPICE_DISCHARGE).any()
assert set(cleaned["readmitted_30"].unique()) == {0, 1}
assert cleaned["age"].cat.ordered

nan_cols = cleaned.isna().sum().loc[lambda s: s > 0]
assert set(nan_cols.index) == set(MISSING_ID_CODES), nan_cols
nan_cols.to_frame("n_missing")

In [ ]:
cleaned[["discharge_group", "admission_source_group", "medical_specialty"]].apply(
    lambda s: s.value_counts().to_dict()
)

Target balance before and after cleaning.


In [ ]:
pd.DataFrame(
    {
        "raw": raw["readmitted"].value_counts(normalize=True),
        "cleaned": cleaned["readmitted"].value_counts(normalize=True),
    }
).round(3)

## 7. Write parquet


In [ ]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
out = DATA_PROCESSED / "diabetic_data.parquet"
cleaned.to_parquet(out, index=False)
print(f"wrote {len(cleaned):,} rows x {cleaned.shape[1]} columns to {out}")

In [ ]:
from eda_diabetes.data import load_processed

round_trip = load_processed()
assert round_trip.shape == cleaned.shape
assert round_trip["age"].cat.ordered
round_trip.head()